# Data Harmonization and Recruitment Requirements

The cross-dataset CNN showed asymmetric generalization. This notebook diagnoses the remaining source shift using the validated acceleration-magnitude windows and turns the findings into a concrete data-collection contract.

This is a planning and audit notebook. It does not change labels, synthesize data, or train another model.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
INTERIM = PROJECT_ROOT / 'data' / 'interim'
magnitude_windows = np.load(PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy', mmap_mode='r')
metadata = pd.read_csv(PROCESSED / 'validated_window_metadata.csv')
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
print('Windows:', magnitude_windows.shape)
print('Windowed participants:', metadata['participant_key'].nunique())


Windows: (18511, 500, 3)
Windowed participants: 284


In [2]:
feature_rows = []
for start in range(0, len(magnitude_windows), 512):
    batch = np.asarray(magnitude_windows[start:start + 512], dtype=np.float32)
    feature_rows.append(pd.DataFrame({
        'window_index': np.arange(start, start + len(batch)),
        'lb_mean': batch[:, :, 0].mean(axis=1),
        'lf_mean': batch[:, :, 1].mean(axis=1),
        'rf_mean': batch[:, :, 2].mean(axis=1),
        'lb_std': batch[:, :, 0].std(axis=1),
        'lf_std': batch[:, :, 1].std(axis=1),
        'rf_std': batch[:, :, 2].std(axis=1),
        'lb_rms': np.sqrt(np.mean(batch[:, :, 0] ** 2, axis=1)),
        'lf_rms': np.sqrt(np.mean(batch[:, :, 1] ** 2, axis=1)),
        'rf_rms': np.sqrt(np.mean(batch[:, :, 2] ** 2, axis=1)),
    }))
window_features = pd.concat(feature_rows, ignore_index=True)
window_features = metadata[['window_id', 'dataset_id', 'participant_key', 'trial_id', 'label']].rename(columns={'window_id': 'window_index'}).merge(window_features, on='window_index')
participant_features = window_features.groupby(['dataset_id', 'participant_key', 'label'], as_index=False).median(numeric_only=True)
print('Participant feature rows:', len(participant_features))


Participant feature rows: 284


In [3]:
feature_names = ['lb_mean', 'lf_mean', 'rf_mean', 'lb_std', 'lf_std', 'rf_std', 'lb_rms', 'lf_rms', 'rf_rms']
shift_rows = []
for feature in feature_names:
    for label in ['healthy', 'stroke']:
        voisard_values = participant_features.loc[(participant_features['dataset_id'] == 'voisard_2025') & (participant_features['label'] == label), feature].dropna()
        felius_values = participant_features.loc[(participant_features['dataset_id'] == 'felius_2024') & (participant_features['label'] == label), feature].dropna()
        if len(voisard_values) and len(felius_values):
            test = mannwhitneyu(voisard_values, felius_values, alternative='two-sided')
            pooled_std = np.sqrt((voisard_values.var(ddof=1) + felius_values.var(ddof=1)) / 2)
            shift_rows.append({
                'feature': feature,
                'label': label,
                'voisard_participants': len(voisard_values),
                'felius_participants': len(felius_values),
                'voisard_median': voisard_values.median(),
                'felius_median': felius_values.median(),
                'median_difference': voisard_values.median() - felius_values.median(),
                'standardized_difference': (voisard_values.mean() - felius_values.mean()) / pooled_std if pooled_std > 0 else np.nan,
                'mannwhitney_p': test.pvalue,
            })
source_shift = pd.DataFrame(shift_rows).sort_values(['label', 'mannwhitney_p'])
print(source_shift.round(3).to_string(index=False))


feature   label  voisard_participants  felius_participants  voisard_median  felius_median  median_difference  standardized_difference  mannwhitney_p
 rf_std healthy                    72                   34           0.712          1.096             -0.384                   -1.087          0.000
 lf_std healthy                    72                   34           0.708          1.047             -0.339                   -1.016          0.000
 rf_rms healthy                    72                   34           1.739          2.136             -0.397                   -0.903          0.000
 lf_rms healthy                    72                   34           1.717          2.057             -0.339                   -0.887          0.000
lf_mean healthy                    72                   34           1.563          1.762             -0.199                   -0.781          0.000
rf_mean healthy                    72                   34           1.582          1.821             -0.2

In [4]:
participant_counts = metadata[['dataset_id', 'participant_key', 'label']].drop_duplicates().groupby(['dataset_id', 'label']).size().reset_index(name='participants')
trial_counts = metadata[['dataset_id', 'participant_key', 'trial_id', 'label']].drop_duplicates().groupby(['dataset_id', 'label']).size().reset_index(name='trials')
window_counts = metadata.groupby(['dataset_id', 'label']).size().reset_index(name='windows')
coverage = participant_counts.merge(trial_counts, on=['dataset_id', 'label']).merge(window_counts, on=['dataset_id', 'label'])
print('Current usable coverage:')
print(coverage.to_string(index=False))


Current usable coverage:
  dataset_id   label  participants  trials  windows
 felius_2024 healthy            34      59     2921
 felius_2024  stroke           129     309    13445
voisard_2025 healthy            72     354     1039
voisard_2025  stroke            49     128     1106


## Required recruitment and harmonization contract

The next recruited clinical dataset should satisfy every item below before it is merged into model training:

1. Real post-stroke participants and healthy controls; simulated stroke recordings remain auxiliary only.
2. Synchronized lower-back, left-foot, and right-foot IMUs, with sensor placement documented by protocol.
3. A recorded sampling rate, synchronized timestamps, and raw tri-axial accelerometer data; gyroscope data should also be retained.
4. Unit metadata and conversion rules, with acceleration converted to g and all signals resampled to 100 Hz.
5. Walking start/stop markers or gait events, so non-walking and turning periods can be excluded.
6. Participant-level age, sex, walking aid, gait speed, clinical deficit side, time since stroke, and clinical severity fields.
7. Multiple walking trials per participant where possible, with participant identifiers preserved across trials.
8. A balanced recruitment design: both labels represented under the same acquisition protocol, rather than adding only more stroke trials to Felius or only more healthy trials to Voisard.
9. A held-out external cohort reserved from the beginning for final evaluation.

Do not use synthetic windows to compensate for the cross-dataset failure until a real-data baseline from a harmonized cohort has been established.

In [5]:
recruitment_spec = pd.DataFrame([
    {'priority': 1, 'requirement': 'Same-protocol real stroke and healthy cohort', 'reason': 'Current cross-dataset CNN fails in the Felius-to-Voisard direction.'},
    {'priority': 2, 'requirement': 'Synchronized LB/LF/RF IMUs', 'reason': 'Matches the current model input and avoids placement confounding.'},
    {'priority': 3, 'requirement': 'Raw signal units, timestamps, and sampling rate', 'reason': 'Required for reproducible 100 Hz preprocessing.'},
    {'priority': 4, 'requirement': 'Walking markers or gait events', 'reason': 'Prevents non-walking windows entering training.'},
    {'priority': 5, 'requirement': 'Age, sex, aid, speed, side, severity, time-since-stroke', 'reason': 'Needed for confound audits and subgroup validation.'},
    {'priority': 6, 'requirement': 'Independent external cohort', 'reason': 'Needed for a credible final generalization estimate.'},
])
source_shift.to_csv(INTERIM / 'source_shift_feature_audit.csv', index=False)
coverage.to_csv(INTERIM / 'usable_data_coverage_summary.csv', index=False)
recruitment_spec.to_csv(INTERIM / 'recruitment_harmonization_spec.csv', index=False)
print('Wrote source_shift_feature_audit.csv')
print('Wrote usable_data_coverage_summary.csv')
print('Wrote recruitment_harmonization_spec.csv')


Wrote source_shift_feature_audit.csv
Wrote usable_data_coverage_summary.csv
Wrote recruitment_harmonization_spec.csv


## Gate for the next modeling round

Pause architecture expansion until a new same-protocol cohort is available or a defensible harmonization procedure is demonstrated. The current CNN remains a strong controlled pilot, but not a source-independent clinical classifier.